# ============================================================
# Author: Mayur Deshmukh
# Name: 01_raw_transform.ipynb
# Project: ML-Binary-Classifier-For-Stock-Price-Prediction
# Purpose: Raw Data Transformation
# Python Version: 3.11
# ============================================================

In [13]:
import pandas as pd
import numpy as np
import os

In [14]:
input_path = os.path.join('..', '..', 'input', 'all_stocks_5yr.csv')
df = pd.read_csv(input_path)
df.head()

,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.12,14.63,14.75,8407500,AAL
1,2013-02-11,14.89,15.01,14.26,14.46,8882000,AAL
2,2013-02-12,14.45,14.51,14.10,14.27,8126000,AAL
3,2013-02-13,14.30,14.94,14.25,14.66,10259500,AAL
4,2013-02-14,14.94,14.96,13.16,13.99,31879900,AAL


In [15]:
# ── Data Cleaning ──────────────────────────────────────────────────────────────

# 1. Parse date and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Name', 'date']).reset_index(drop=True)

# 2. Drop exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {before - len(df)}")

# 3. Missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Drop rows where price or volume data is missing
df = df.dropna(subset=['open', 'high', 'low', 'close', 'volume'])

# 4. Remove rows with non-positive prices or volume
invalid_mask = (df[['open', 'high', 'low', 'close']] <= 0).any(axis=1) | (df['volume'] < 0)
print(f"\nInvalid price/volume rows removed: {invalid_mask.sum()}")
df = df[~invalid_mask].reset_index(drop=True)

# 5. Sanity check: high >= low
inconsistent = df[df['high'] < df['low']]
print(f"Rows where high < low: {len(inconsistent)}")
df = df[df['high'] >= df['low']].reset_index(drop=True)

# 6. Correct dtypes
df['volume'] = df['volume'].astype('int64')
df['Name'] = df['Name'].astype('category')

print(f"\nCleaned dataset shape: {df.shape}")
df.dtypes

Duplicates removed: 0

Missing values per column:
date       0
open      11
high       8
low        8
close      0
volume     0
Name       0
dtype: int64

Invalid price/volume rows removed: 0
Rows where high < low: 1

Cleaned dataset shape: (619028, 7)


date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
Name            category
dtype: object

In [16]:
# ── Target Label: Price Direction ──────────────────────────────────────────────

# Compare each day's close to the previous day's close, grouped by ticker
df['target'] = (df.groupby('Name')['close']
                  .diff()
                  .gt(0)
                  .astype(int))

# First row per ticker has no previous day — drop it
df = df.dropna(subset=['target']).reset_index(drop=True)

print(f"Label distribution:\n{df['target'].value_counts()}")
print(f"\nLabel balance: {df['target'].mean():.2%} up days")
df[['date', 'Name', 'close', 'target']].head(10)

Label distribution:
target
1    322446
0    296582
Name: count, dtype: int64

Label balance: 52.09% up days


,date,Name,close,target
0,2013-02-08,A,45.08,0
1,2013-02-11,A,44.60,0
2,2013-02-12,A,44.62,1
3,2013-02-13,A,44.75,1
4,2013-02-14,A,44.58,0
5,2013-02-15,A,42.25,0
6,2013-02-19,A,43.01,1
7,2013-02-20,A,42.24,0
8,2013-02-21,A,41.63,0
9,2013-02-22,A,41.80,1


In [17]:
# ── Label Distribution per Ticker ──────────────────────────────────────────────

label_dist = (df.groupby('Name')['target']
                .value_counts(normalize=True)
                .mul(100)
                .round(2)
                .rename('percentage')
                .reset_index())

label_dist['target'] = label_dist['target'].map({1: 'Up', 0: 'Down'})
label_dist = label_dist.pivot(index='Name', columns='target', values='percentage').reset_index()
label_dist.columns.name = None

print(f"Label distribution (%) per ticker — showing top 5:")
display(label_dist.head())

print(f"\nFull dataset (top 5 rows):")
display(df.head())

Label distribution (%) per ticker — showing top 5:


,Name,Down,Up
0,A,47.74,52.26
1,AAL,47.26,52.74
2,AAP,49.56,50.44
3,AAPL,48.37,51.63
4,ABBV,46.47,53.53



Full dataset (top 5 rows):


,date,open,high,low,close,volume,Name,target
0,2013-02-08,45.07,45.35,45.00,45.08,1824755,A,0
1,2013-02-11,45.17,45.18,44.45,44.60,2915405,A,0
2,2013-02-12,44.81,44.95,44.50,44.62,2373731,A,1
3,2013-02-13,44.81,45.24,44.68,44.75,2052338,A,1
4,2013-02-14,44.72,44.78,44.36,44.58,3826245,A,0


In [18]:
# ── Filter: Top N Stocks by Record Count (max 25K total) ───────────────────────

record_counts = df.groupby('Name').size().sort_values(ascending=False)

top10 = record_counts.head(10)
top9  = record_counts.head(9)

if top10.sum() <= 25000:
    selected_tickers = top10.index.tolist()
    print(f"Top 10 stocks selected (total records: {top10.sum()})")
else:
    selected_tickers = top9.index.tolist()
    print(f"Top 10 exceeded 25K records ({top10.sum()}), falling back to top 9 (total records: {top9.sum()})")

print(f"\nSelected tickers: {selected_tickers}")
print(f"\nRecord counts:")
print(record_counts[selected_tickers])

df_filtered = df[df['Name'].isin(selected_tickers)].reset_index(drop=True)

print(f"\nFiltered dataset shape: {df_filtered.shape}")
df_filtered.head()

Top 10 stocks selected (total records: 12590)

Selected tickers: ['ZTS', 'A', 'AAL', 'AAP', 'AAPL', 'ABBV', 'ABC', 'ABT', 'ACN', 'WMT']

Record counts:
Name
ZTS     1259
A       1259
AAL     1259
AAP     1259
AAPL    1259
ABBV    1259
ABC     1259
ABT     1259
ACN     1259
WMT     1259
dtype: int64

Filtered dataset shape: (12590, 8)


,date,open,high,low,close,volume,Name,target
0,2013-02-08,45.07,45.35,45.00,45.08,1824755,A,0
1,2013-02-11,45.17,45.18,44.45,44.60,2915405,A,0
2,2013-02-12,44.81,44.95,44.50,44.62,2373731,A,1
3,2013-02-13,44.81,45.24,44.68,44.75,2052338,A,1
4,2013-02-14,44.72,44.78,44.36,44.58,3826245,A,0


In [19]:
# ── Export Filtered Dataset ─────────────────────────────────────────────────────

output_path = os.path.join('..', '..', 'output', 'filtered_stocks.csv')
df_filtered.to_csv(output_path, index=False)
print(f"Exported {df_filtered.shape[0]} rows to: {output_path}")

Exported 12590 rows to: ..\..\output\filtered_stocks.csv
